In [16]:
import os
import pandas as pd
import lyricsgenius
from datetime import datetime
import torch
from transformers import pipeline
from tqdm import tqdm
import streamlit as st
import plotly.graph_objects as go
import numpy as np


In [39]:
def build_kpop_lyric_database():
    """
    Scrapes bilingual K-pop lyrics via the Genius API, structuring the 
    raw cultural text into a relational dataset mapped by record label 
    for strategic sentiment analysis.
    """
    # 1. Authenticate with the Genius API
    GENIUS_TOKEN = "a1LaCzSH2tNrKjhBPMBZgJ9D_kkaoZdVyCpuRtZT1lSptH4ionBsrEAeFO7j01h3"

    genius = lyricsgenius.Genius(GENIUS_TOKEN)

    # Apply settings as properties
    genius.verbose = False
    genius.remove_section_headers = True
    genius.skip_non_songs = True

    # Add network stability
    genius.timeout = 30  
    genius.retries = 3   

    corporate_matrix = {
        "SM Entertainment": ["aespa", "NCT 127", "Red Velvet"],
        "HYBE": ["BTS", "NewJeans", "LE SSERAFIM"],
        "YG Entertainment": ["BLACKPINK", "TREASURE", "BABYMONSTER"],
        "JYP Entertainment": ["Stray Kids", "TWICE", "ITZY"]
    }
    
    # Parameters for the scrape
    songs_per_artist = 15  # Kept at 1 for a quick test run
    dataset_rows = []
    
    print("--- Initiating Corporate Lyric Scraping Pipeline ---")

    # 3. Execute the Scrape
    for label, artists in corporate_matrix.items():
        print(f"\nAnalyzing Label: {label}")
        
        for artist_name in artists:
            print(f"  -> Pulling top {songs_per_artist} tracks for {artist_name}...")
            
            try:
                artist_data = genius.search_artist(
                    artist_name, 
                    max_songs=songs_per_artist, 
                    sort="popularity"
                )
                
                if artist_data is None:
                    print(f"     [!] Could not locate artist: {artist_name}")
                    continue
                
                for song in artist_data.songs:
                    if not song.lyrics or "Instrumental" in song.title:
                        continue
                        
                    clean_lyrics = song.lyrics.replace("Embed", "").strip()
                    
                    dataset_rows.append({
                        "record_label": label,
                        "artist": artist_name,
                        "song_title": song.title,
                        "release_year": song.year[:4] if getattr(song, 'year', None) else "Unknown",
                        "page_views": song.to_dict().get("stats", {}).get("pageviews", 0),
                        "raw_lyrics": clean_lyrics
                    })
                    
            except Exception as e:
                print(f"     [!] Error processing {artist_name}: {e}")
                
    return dataset_rows

# --- MAIN EXECUTION BLOCK ---

# 1. Run the scrape
scraped_data = build_kpop_lyric_database()
print(f"\nScrape complete. Collected {len(scraped_data)} songs.")

# 2. Structure the Data
print("\n--- Scraping Complete. Structuring Data ---")
df = pd.DataFrame(scraped_data)

if not df.empty:
    # Temporarily comment out the filter so we keep the lyrics
    # df = df[df["release_year"] != "Unknown"]

    # Sort the final dataset logically
    df = df.sort_values(by=["record_label", "artist", "page_views"], ascending=[True, True, False])

    # 3. Export to CSV
    output_dir = "./data/raw_lyrics"
    os.makedirs(output_dir, exist_ok=True)
        
    timestamp = datetime.now().strftime("%Y%m%d")
    output_file = f"{output_dir}/kpop_corporate_lyrics_{timestamp}.csv"
        
    df.to_csv(output_file, index=False, encoding='utf-8-sig')
    print(f"Dataset successfully compiled! Saved to: {output_file}")
    print(f"Total tracks ready for NLP pipeline: {len(df)}")
    
else:
    print("Dataset is empty. No CSV was generated.")

# Display the first few rows in Jupyter
df.head()

--- Initiating Corporate Lyric Scraping Pipeline ---

Analyzing Label: SM Entertainment
  -> Pulling top 15 tracks for aespa...
  -> Pulling top 15 tracks for NCT 127...
  -> Pulling top 15 tracks for Red Velvet...

Analyzing Label: HYBE
  -> Pulling top 15 tracks for BTS...
  -> Pulling top 15 tracks for NewJeans...
  -> Pulling top 15 tracks for LE SSERAFIM...

Analyzing Label: YG Entertainment
  -> Pulling top 15 tracks for BLACKPINK...
  -> Pulling top 15 tracks for TREASURE...
  -> Pulling top 15 tracks for BABYMONSTER...

Analyzing Label: JYP Entertainment
  -> Pulling top 15 tracks for Stray Kids...
  -> Pulling top 15 tracks for TWICE...
  -> Pulling top 15 tracks for ITZY...

Scrape complete. Collected 180 songs.

--- Scraping Complete. Structuring Data ---
Dataset successfully compiled! Saved to: ./data/raw_lyrics/kpop_corporate_lyrics_20260919.csv
Total tracks ready for NLP pipeline: 180


,record_label,artist,song_title,release_year,page_views,raw_lyrics
45,HYBE,BTS,Dynamite,Unknown,4796869,"'Cause I, I, I'm in the stars tonight\nSo watc..."
46,HYBE,BTS,Butter,Unknown,3734149,Smooth like butter\nLike a criminal undercover...
47,HYBE,BTS,FAKE LOVE,Unknown,1435474,널 위해서라면 난\n슬퍼도 기쁜 척 할 수가 있었어\n널 위해서라면 난\n아파도 강...
48,HYBE,BTS,MIC Drop (Steve Aoki Remix),Unknown,1286341,"Yeah, 누가 내 수저 더럽대\nI don't care, 마이크 잡음 금수저 여럿..."
49,HYBE,BTS,Permission to Dance,Unknown,1203127,It's the thought of being young\nWhen your hea...


In [41]:
def analyze_kpop_sentiment(data_path=None):
    """
    Processes bilingual (Korean/English) K-pop lyrics using an open-source
    multilingual Transformer model to extract cross-lingual sentiment metrics.
    """
    
    # 1. Initialize the Multilingual Sentiment Pipeline
    # We use a model fine-tuned on multilingual text (XLM-RoBERTa base architecture)
    model_name = "tabularisai/multilingual-sentiment-analysis"
    
    # Check for CUDA availability to leverage a GPU if available
    device = 0 if torch.cuda.is_available() else -1
    print(f"Initializing model on device: {'GPU (cuda)' if device == 0 else 'CPU'}")
    
    try:
        sentiment_pipeline = pipeline(
            "sentiment-analysis", 
            model=model_name, 
            device=device
        )
        print("Model loaded successfully.")
    except Exception as e:
        print(f"Failed to load the Hugging Face pipeline. Error: {e}")
        return

    # 2. Sample Dataset (Simulating your structured lyrics database)
    # Replace this block with pd.read_csv(data_path) when your data is ready
        df = pd.read_csv("./data/raw_lyrics/kpop_corporate_lyrics_20260919.csv")

    # 3. Batch Processing Loop
    # Storing results in lists for rapid DataFrame appending
    final_labels = []
    confidence_scores = []
    
    print("\n--- Processing bilingual lyrics through Transformer vector space ---")
    
    # tqdm adds a clean progress bar, essential for tracking longer dataset loops asynchronously
    for lyric_text in tqdm(df["lyrics"], desc="Analyzing tracks"):
        try:
            # We truncate long text to the model's max token limit (typically 512 tokens)
            result = sentiment_pipeline(lyric_text, truncation=True)[0]
            
            final_labels.append(result["label"])
            confidence_scores.append(round(result["score"], 4))
            
        except Exception as e:
            final_labels.append("ERROR")
            confidence_scores.append(0.0)
            print(f"Error processing text snippet: {lyric_text[:30]}... Error: {e}")

    # 4. Integrate Results back into the Strategic Dataset
    df["sentiment_category"] = final_labels
    df["confidence_score"] = confidence_scores

    # 5. Export for Business Intelligence Analysis
    # This structured output is what feeds your dashboard or K-Means clustering steps
    output_dir = "./data/processed_analytics"
    os.makedirs(output_dir, exist_ok=True)
    
    output_file = f"{output_dir}/kpop_label_sentiment.csv"
    df.to_csv(output_file, index=False)
    print(f"\nPipeline complete! Analytical dataset saved to: {output_file}")
    
    return df

if __name__ == "__main__":
    # Run with sample data to verify architecture works locally
    processed_df = analyze_kpop_sentiment()
    
    # Quick glimpse of the engineered features
    print("\nSample Output Preview:")
    print(processed_df[["label", "artist", "track", "sentiment_category", "confidence_score"]])



Initializing model on device: CPU


config.json:   0%|          | 0.00/851 [00:00<?, ?B/s]

C:\Users\barre\my_zensvi\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\barre\.cache\huggingface\hub\models--tabularisai--multilingual-sentiment-analysis. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors:   0%|          | 0.00/541M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

Model loaded successfully.

--- Processing bilingual lyrics through Transformer vector space ---


UnboundLocalError: cannot access local variable 'df' where it is not associated with a value

In [42]:
def analyze_kpop_sentiment(data_path="./data/raw_lyrics/kpop_corporate_lyrics_20260919.csv"):
    """
    Processes bilingual (Korean/English) K-pop lyrics using an open-source
    multilingual Transformer model to extract cross-lingual sentiment metrics.
    """
    
    # 1. Initialize the Multilingual Sentiment Pipeline
    model_name = "tabularisai/multilingual-sentiment-analysis"
    
    # Check for CUDA availability to leverage a GPU if available
    device = 0 if torch.cuda.is_available() else -1
    print(f"Initializing model on device: {'GPU (cuda)' if device == 0 else 'CPU'}")
    
    try:
        sentiment_pipeline = pipeline(
            "sentiment-analysis", 
            model=model_name, 
            device=device
        )
        print("Model loaded successfully.")
    except Exception as e:
        print(f"Failed to load the Hugging Face pipeline. Error: {e}")
        return

    # 2. Load the dataset
    df = pd.read_csv(data_path)

    # 3. Batch Processing Loop
    final_labels = []
    confidence_scores = []
    
    print("\n--- Processing bilingual lyrics through Transformer vector space ---")
    
    # Use the correct column name: 'raw_lyrics'
    for lyric_text in tqdm(df["raw_lyrics"], desc="Analyzing tracks"):
        try:
            # We truncate long text to the model's max token limit (typically 512 tokens)
            # Ensure the text is treated as a string to prevent float errors on empty rows
            result = sentiment_pipeline(str(lyric_text), truncation=True)[0]
            
            final_labels.append(result["label"])
            confidence_scores.append(round(result["score"], 4))
            
        except Exception as e:
            final_labels.append("ERROR")
            confidence_scores.append(0.0)
            print(f"Error processing text snippet: {str(lyric_text)[:30]}... Error: {e}")

    # 4. Integrate Results back into the Strategic Dataset
    df["sentiment_category"] = final_labels
    df["confidence_score"] = confidence_scores

    # 5. Export for Business Intelligence Analysis
    output_dir = "./data/processed_analytics"
    os.makedirs(output_dir, exist_ok=True)
    
    output_file = f"{output_dir}/kpop_label_sentiment.csv"
    df.to_csv(output_file, index=False, encoding='utf-8-sig')
    print(f"\nPipeline complete! Analytical dataset saved to: {output_file}")
    
    return df

if __name__ == "__main__":
    # Run the analysis
    processed_df = analyze_kpop_sentiment()
    
    # Quick glimpse using the correct column names
    print("\nSample Output Preview:")
    print(processed_df[["record_label", "artist", "song_title", "sentiment_category", "confidence_score"]].head())

Initializing model on device: CPU


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Model loaded successfully.

--- Processing bilingual lyrics through Transformer vector space ---


Analyzing tracks: 100%|██████████████████████████████████████████████████████████████| 180/180 [00:19<00:00,  9.02it/s]


Pipeline complete! Analytical dataset saved to: ./data/processed_analytics/kpop_label_sentiment.csv

Sample Output Preview:
  record_label artist                   song_title sentiment_category  \
0         HYBE    BTS                     Dynamite      Very Positive   
1         HYBE    BTS                       Butter      Very Positive   
2         HYBE    BTS                    FAKE LOVE      Very Positive   
3         HYBE    BTS  MIC Drop (Steve Aoki Remix)      Very Positive   
4         HYBE    BTS          Permission to Dance      Very Positive   

   confidence_score  
0            0.8425  
1            0.7872  
2            0.8055  
3            0.5693  
4            0.3669  
